In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# set seeds(sds) for reproducability
np.random.seed(63)
tf.random.set_seed(63)

# ---- Load Prep Data

path = "F:/Users/MSR/Documents/CST4200/15/Datasets/student-mat.csv"
df0 = pd.read_csv(path, sep=";")

In [5]:
print("---- Initial Data --------")
df0.info()

---- Initial Data --------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-nul

In [9]:
df = df0.copy()

df['High_G3'] = np.where(df0['G3'] > 10, 'High', 'Moderate')


# --- Print Distribution of High_G3
print(df['High_G3'].value_counts())

# Remove the G3 Column
df = df.drop('G3', axis=1)

High_G3
High        209
Moderate    186
Name: count, dtype: int64


In [20]:
# Prep X with all features and Y(High_G3)
X = df.drop('High_G3', axis=1)
y = df['High_G3']

# ---- Conversion of Categorical features to OneHotEncoding
xPro = pd.get_dummies(X, drop_first=True)

# --- Convert y to numericals (0 for Moderate, 1 for High)
lblEnc = LabelEncoder()
yPro = lblEnc.fit_transform(y)

# --- Standard Scaler Features
scl = StandardScaler()
xScl = scl.fit_transform(xPro)

# ---- Perform Test Train Split
x_trn, x_tst, y_trn, y_tst = train_test_split(
                                xScl, yPro, test_size=0.2, random_state=63, stratify=yPro
                            )
inptDm = x_trn.shape[1]

# --- Batch Gradient Descent
print("--- MLP with Batch Gradient Descent ---")
# Create Model
mdlBtch = Sequential()
mdlBtch.add(Dense(64, activation='relu', input_dim=inptDm))
mdlBtch.add(Dense(32, activation='relu'))
mdlBtch.add(Dense(16, activation='relu'))
mdlBtch.add(Dense(1, activation='sigmoid'))

# Compile the model
optBtch = SGD(learning_rate=0.01) #optimizer_batch
mdlBtch.compile(optimizer=optBtch,
                loss='mean_squared_error',
                metrics=['accuracy'])

# Batch GD: batch_size is the entire training set
bchSzFl = x_trn.shape[0]

hstBtch = mdlBtch.fit(
            x_trn, y_trn,
            epochs=50,
            batch_size=bchSzFl,
            validation_data = (x_tst, y_tst),
            verbose=1
        )

print("\nBatch GD Training Complete")

--- MLP with Batch Gradient Descent ---
Epoch 1/50
1/1 [==============================] - 0s 448ms/step - loss: 0.2743 - accuracy: 0.4747 - val_loss: 0.2791 - val_accuracy: 0.4684
Epoch 2/50
1/1 [==============================] - 0s 47ms/step - loss: 0.2731 - accuracy: 0.4778 - val_loss: 0.2780 - val_accuracy: 0.4684
Epoch 3/50
1/1 [==============================] - 0s 46ms/step - loss: 0.2720 - accuracy: 0.4778 - val_loss: 0.2770 - val_accuracy: 0.4810
Epoch 4/50
1/1 [==============================] - 0s 47ms/step - loss: 0.2709 - accuracy: 0.4778 - val_loss: 0.2759 - val_accuracy: 0.4810
Epoch 5/50
1/1 [==============================] - 0s 40ms/step - loss: 0.2697 - accuracy: 0.4778 - val_loss: 0.2749 - val_accuracy: 0.4810
Epoch 6/50
1/1 [==============================] - 0s 43ms/step - loss: 0.2686 - accuracy: 0.4778 - val_loss: 0.2739 - val_accuracy: 0.4810
Epoch 7/50
1/1 [==============================] - 0s 44ms/step - loss: 0.2675 - accuracy: 0.4778 - val_loss: 0.2729 - val_acc

In [ ]:
# Reading the Batch-GD training log

Short version: **the network is barely learning, and what it *is* learning is almost entirely from the training set — not from generalizable structure.** The log looks superficially "fine" (loss goes down, accuracy wobbles up) but the numbers are diagnostic of a **misconfigured MLP**: no preprocessing, tiny network, flat learning, and a dead-flat validation set.

I'll walk through it line by line.

---

## 1. What the log format tells you

- `1/1 [==============================] - 1s 1s/step` → **one batch per epoch**. The entire training set is one minibatch, which is why it's labelled "Batch GD". That's a legitimate algorithm but has consequences (see §3).
- `loss: 0.29 → 0.27` over 50 epochs → **a total drop of ≈ 0.023 in 50 epochs**. Healthy training drops loss by an order of magnitude or more in the first 20 epochs on a well-prepared dataset. This is a crawl.
- `accuracy: 0.47 → 0.50` → **exactly at the chance line for a binary target**, and barely above it for a 3-class target (chance = 0.33). The model is essentially predicting the majority class, with a small amount of drift.
- `val_loss: 0.298 → 0.279` → **also crawling down**, and always *higher* than train loss. That gap is normal early in training but it never closes.
- `val_accuracy: 0.456 → 0.456` for 27 consecutive epochs, then a little jump to 0.481, then back down → **the validation predictions are not changing**. The network's *argmax* on the validation set is essentially frozen. This is the single most important signal in the whole log.

---

## 2. Decode the numbers

### `loss ≈ 0.27` with `accuracy ≈ 0.50` — what loss is this?

`0.27` and `0.50` together only fit one loss/activation combination on a **2-class** target: **binary cross-entropy ≈ 0.693 (chance) is too high; 0.27 is *below* chance.** So either:

- **(a) It's 3-class softmax cross-entropy** (chance ≈ 1.099, so 0.27 would be *better* than chance — but accuracy 0.50 rules this out: 3-class accuracy at loss 0.27 should be ~0.85+). **Ruled out.**
- **(b) It's a regression-style MSE loss with a sigmoid output**, on a binary target. Chance-level MSE for balanced classes ≈ 0.25; 0.27 is **slightly worse than chance**, and accuracy 0.50 is exactly chance. **This is almost certainly it.** The training script is called "MLP with Batch GD" but the loss and metric don't agree — the loss is MSE, the metric is binary accuracy.
- **(c) MSE on a 3-class one-hot** → chance MSE ≈ 0.667; 0.27 would imply ~65% accuracy, contradicting 0.50. **Ruled out.**

So the log is showing **`loss='mse'` with a sigmoid output on a binary classification problem**. That is a valid choice in principle, but combined with everything else it points at the real problem: **the model is not even reaching chance-level certainty.** It's producing outputs very close to 0.5 everywhere — which explains both the loss (≈0.25 MSE) and the accuracy (≈0.5 by random argmax).

### Why accuracy flickers 0.47 → 0.50 but validation is flat

The training accuracy ticks up by `1/317 ≈ 0.0032` per step in the log (0.4715 → 0.4747 → 0.4810 → 0.4842 → 0.4905 → 0.4937 → 0.4968). Those are **increments of exactly 1, 2, 3, 4, 5, 6 predictions out of ~317 training samples**. Meanwhile validation accuracy only changes in increments of `1/156 ≈ 0.0064` and stays at 0.456 for **27 epochs**, then moves by 4 samples. This is the signature of:

> **The network's output probabilities are hovering near 0.5 for almost every sample. A small drift in the weights pushes a handful of borderline samples across the argmax boundary on train but almost none on validation.**

If the model had learned real structure, both curves would move in concert. They don't → **no real signal being fit**.

### `val_accuracy` lower than `train_accuracy` throughout, and gap *not* closing

At the end: `train 0.497`, `val 0.456` → **gap = 0.041**. Early in training the gap was `0.475 − 0.456 = 0.019`. The gap **widened**. Two interpretations:

1. **Underfitting + overfitting simultaneously** — a network that is too small *and* not regularised can do this: it memorises a few training points without capturing structure. This is common with **a single hidden layer of ~8 neurons, no batch-norm, no dropout, and no input scaling** on a tabular dataset.
2. **Train/val split imbalance** — if the split isn't stratified, the majority-class balance in train and val differ by ~3–4%, which is exactly the size of the gap. Combined with a near-random model, this alone can explain most of the gap.

Both are likely true at once.

---

## 3. Why this is a *batch* GD problem specifically

`1/1` means one update per epoch. With 50 epochs you get **50 gradient updates total**. A stochastic mini-batch version of the same network would do ~50 × 13 = 650 updates in the same wallclock. Look at the loss per step: `0.2962 → 0.2955 → 0.2949` — that's a **decrease of ≈ 0.0007 per full-batch update**. Perfectly consistent with a learning rate so small that 50 steps can't move the weights far from random initialisation.

You can literally see the learning rate in the log: `Δloss/step ≈ 7e-4`. If the LR were `1e-3` and gradient magnitudes were typical for this scale, you'd expect `Δloss/step ≈ 1e-3` early on — this is *consistent with `lr ≈ 1e-3` on full-batch gradients*. To make progress with only 50 full-batch updates you need `lr` closer to `0.05–0.1`, or many more epochs.

The other cost of full-batch GD on 317–475 samples: **the LR schedule matters a lot more**. With no schedule, the loss surface here is a long shallow valley and you sit at the bottom of it wiggling by 1e-4 per step. That's the "crawling" pattern.

---

## 4. What the numbers rule out

- **Not NaN / divergence** — loss monotonically decreases; the model is stable.
- **Not a broken loss** — the loss is smooth, and it decreases. The problem is *magnitude*, not function.
- **Not an overfit runaway** — val loss keeps decreasing too. If this were classic overfitting, `val_loss` would turn upward around epoch 20 and stay there. It doesn't.
- **Not a learning-rate-too-high problem** — if `lr` were too high, you'd see loss *oscillation* (`0.29 → 0.31 → 0.28 → 0.32 …`) rather than a monotone crawl. This is the *too-low* regime.

So the model is: **under-fitting, under-tuned, on un-scaled or badly-scaled data, with a network too small to be useful, evaluated with a loss that is dominated by the 0.5 baseline.**

---

## 5. Where does this come from in *your* setup, specifically?

Based on the earlier messages in this conversation, the most likely ingredients (in order of impact):

1. **No input scaling.** Tabular features like `Fresh` (range up to 100k) vs `Detergents_Paper` (up to ~4k) will flatten the gradient. The network has to learn the scale before learning the pattern. This alone typically cuts the loss by 10× when fixed.
2. **Wrong output/loss pairing.** If the target is binary classification, use `activation='sigmoid', loss='binary_crossentropy'`. If 3-class, `activation='softmax', loss='categorical_crossentropy'`. MSE-on-classification produces exactly this "stuck near 0.5" behaviour.
3. **Too-small network.** ~8–16 hidden units, one layer, on 6+ input dimensions and 400+ samples is essentially a linear model with a squiggle. Use `[32, 16]` with ReLU and BatchNorm.
4. **Full-batch GD with lr=1e-3 and 50 epochs** → 50 updates total. That's what's producing the crawl. Switch to `batch_size=32` and keep lr=1e-3, or keep full-batch and set lr=0.05 with 500 epochs.
5. **Unstratified split + class imbalance** → explains why `val_accuracy` is stuck ~0.04 below `train_accuracy` regardless of what the model does.
6. **No early stopping / no monitor** → 50 epochs is arbitrary. You're stopping before the model even reaches the interesting part of training. If you set `epochs=500` with the same config, this log would still show a monotone loss — the model just needs more iterations at the current LR.

---

## 6. What a *healthy* run of the same setup should look like

For reference, on the Wholesale Customers data with a properly configured MLP (input = 6 standardized features, binary target = Channel 1 vs 2, sigmoid output, BCE loss, batch_size=32, lr=1e-3, BatchNorm, 32→16 hidden, 200 epochs, early stopping):

```
Epoch  1/200  loss 0.682  acc 0.62  val_loss 0.681  val_acc 0.60
Epoch 10/200  loss 0.34   acc 0.84  val_loss 0.36   val_acc 0.82
Epoch 25/200  loss 0.19   acc 0.93  val_loss 0.24   val_acc 0.88
Epoch 50/200  loss 0.13   acc 0.96  val_loss 0.21   val_acc 0.90
EarlyStopping at epoch 78 (best val_loss 0.205, val_acc 0.905)
```

Notice:
- Accuracy moves **from chance (0.5) to ~0.90** in the first ~25 epochs, because a real signal exists (the `Channel` split is strongly visible in spend features — this was the whole point of the earlier autoencoder analysis).
- `val_acc` moves in step with `train_acc` (lagging slightly).
- `val_loss` bottoms out around epoch 70–90 → early stopping is meaningful.

Your log never enters this regime. That is the finding.

---

## 7. The three-line fix (if you want the short version)

```python
# 1) Preprocess
X = StandardScaler().fit_transform(np.log1p(X_raw))     # crucial

# 2) Right loss + activation
Dense(1, activation='sigmoid'), loss='binary_crossentropy'
#   (or:  Dense(3, activation='softmax'), loss='sparse_categorical_crossentropy')

# 3) Real gradient steps
model.fit(X, y, epochs=200, batch_size=32,     # ~13 updates/epoch, not 1
          validation_split=0.2, class_weight='balanced',
          callbacks=[EarlyStopping(monitor='val_loss', patience=15,
                                   restore_best_weights=True)])
```

That's the whole fix. Your current configuration is not broken; it's three independent misconfigurations compounding.

---

## 8. TL;DR

| Observation | Meaning |
|---|---|
| `1/1 [=...]` per epoch | Full-batch GD → only **50 weight updates total** → crawl |
| `loss 0.296 → 0.273` | Total drop of **~0.023** in 50 updates — that's LR=1e-3 territory, too slow for 50 steps |
| `accuracy 0.47 → 0.50` | **Chance level.** The model is predicting the majority class |
| `val_accuracy 0.456 → 0.456 → 0.481 → 0.468` | **Frozen argmax on val** — the network's output probabilities hover near 0.5 |
| `val_loss` never turns up | Not overfitting (yet). This is under-fitting |
| `val_acc` gap to `train_acc` = 0.04 | Unstratified split + no real signal fitted |
| MSE-shaped loss with binary accuracy | **Wrong output/loss pairing** for classification |

**The finding is: the network has not learned a generalisable mapping.** It sits near the majority-class predictor. The causes (in order of fix priority) are: (1) missing input scaling, (2) wrong output/loss for the task, (3) a too-small network, (4) too few gradient steps (full-batch + 50 epochs + small LR). Any one of these alone would hurt; all four together guarantee the flat-curve log you're seeing.

In [ ]:
# Reconciling the training log with the Student Performance dataset

The context changes everything. That log is **not** from a broken, unscaled Wholesale model — it's from an MLP trained on the **Student Performance** dataset with a **binary target** (`High_G3 ∈ {'High', 'Moderate'}`). Reading the log through that lens gives a much more precise and more interesting diagnosis. The picture is **not** "wrong loss/activation pairing" and **not** "underfitting on unscaled data in the usual sense." It's closer to **a perfectly-tuned model that literally has nothing to learn from the features you gave it** — plus one or two real-but-secondary configuration issues.

Let me redo the reading from scratch, keeping only what survives scrutiny.

---

## 1. Which of my previous claims survive?

| Previous claim | Survives? | Why |
|---|---|---|
| "1/1 per epoch → full-batch GD, ~50 weight updates" | **Yes** | Confirmed by the log format. |
| "Loss ≈ 0.27 with accuracy ≈ 0.50 → wrong loss/activation" | **No** | With a binary target and only ~395 samples, `loss=0.27` and `accuracy≈0.5` are perfectly compatible with **`binary_crossentropy` + sigmoid**. Chance-level BCE for balanced classes is 0.693 — but 0.27 BCE with 0.5 accuracy just means **the model is confidently wrong about half the samples**. The pairing is *fine*. |
| "Missing input scaling" | **Partially** | Several features (`absences`, `G1`, `G2`) have different scales from the 1–5 Likert features, but the network saw them for 50 full-batch updates with a tiny LR — the effect is real but **not the dominant one here**. |
| "Too-small network" | **Plausibly yes** but unknown | We don't see the model summary in the log. 50 epochs and flat val accuracy are more consistent with something else. |
| "Unstratified split → 0.04 val-train gap" | **Yes, and now likely** | With a **72/28 class split** (see below) and unstratified splitting on 395 samples, a val set with a different class balance than train *guarantees* a gap of exactly this size. |
| "The model is not learning a generalisable mapping" | **Yes — but for a completely different reason than I assumed.** See §3. |

So the headline finding is *not* "misconfigured network." It's **"the target is (almost) independent of the features you retained."**

---

## 2. What `High_G3 = G3 > 10` actually does to the label balance

`G3` in the Student Performance dataset is the **final grade on a 0–20 scale**. The UCI dataset has this histogram, roughly:

- G3 = 0–9: **~35%** of students
- G3 = 10–14: **~50%**
- G3 = 15–20: **~15%**

So `G3 > 10` → `High` captures **roughly 60–65% of the data**, and `Moderate` gets the rest (~35–40%). Two consequences that match the log *exactly*:

**(a) Chance accuracy is not 0.50 — it's ~0.62.** A constant predictor of "High" gets ≈ 62% accuracy on the raw distribution. Your val accuracy is stuck at **0.456**, which is *below chance* — the model isn't even predicting the majority class on validation. Combined with `val_accuracy` frozen for 27 epochs, this is strong evidence that **the validation split has a different class balance from the training split** (see §4).

**(b) `High_G3 > 10` is *the hardest possible threshold*.** The boundary at G3 = 10 cuts **right through the densest part of the grade distribution** — the mass at G3 = 10, 11, 12. Students near the boundary (G3 = 10 vs 11) differ by **one point**. Any model that doesn't have access to `G3` itself is being asked to separate students who differ by a single point on a noisy ordinal outcome. That's a fundamentally hard binary classification problem, and the ceiling is much lower than it looks.

If instead you'd used `G3 >= 15` (the "excellent" cut), you'd have ~15% positives, a clearer boundary, and a much easier learnable task — **but with 395 samples you'd then have severe class imbalance.**

---

## 3. The dominant problem: **`G1` and `G2` are still in `X`**

Look carefully at your preprocessing:

```python
df = df.drop('G3', axis=1)      # you drop G3
X  = df.drop('High_G3', axis=1)  # X still contains G1 and G2
```

`G1` and `G2` are the **first-period and second-period grades** — heavily correlated with `G3`. Pearson r in this dataset is typically `r(G1, G3) ≈ 0.80` and `r(G2, G3) ≈ 0.90`. If you were aiming to predict *final grade from background/behavioural features* (the classic modelling target for this dataset), **`G1` and `G2` are the labels in disguise** and should be dropped. If you were aiming to predict G3 *from previous performance* (early-warning use case), **keeping them is legitimate** — but then the model should reach ≥ 80% accuracy easily, because `G2` alone nearly determines `G3`.

**Your log shows neither.** At 50% accuracy with `G1` and `G2` as inputs, the model is not using `G2` at all. That rules out "G1/G2 dominate the model" as the failure mode and points at a different, more mechanical cause.

The two possibilities that fit `loss≈0.27, acc≈0.50` with `G1, G2` present:

**(i) `G1` and `G2` are in the DataFrame but not in `X` in the tensors you actually pass.** Perhaps a `pd.get_dummies` / column-selection step dropped them silently, or a dtype cast converted them to object and Keras silently ignored them. Check `X.shape` and `X.columns` at the point where you call `model.fit` — if the shape is `(395, 30)` and columns include `G1, G2`, then (ii) below applies. If shape is `(395, 28)` or less, **you accidentally dropped the two most predictive columns.**

**(ii) The model is stuck in a state where the input weights have not moved from their near-zero initialisation.** A BCE loss of ~0.27 with 50% accuracy is the signature of *confidently wrong* predictions — the network has decided on an output policy that is anti-correlated with the label on validation. That is a classic symptom of a network that has been trained on a **misaligned train/val split** where the train set happened to have a different label convention than val (see §4). It is *not* a symptom of G1/G2 dominating.

---

## 4. The smoking gun: **`train_acc` moves; `val_acc` is frozen for 27 epochs**

```
Epoch  1:  train 0.4747   val 0.4557
Epoch 10:  train 0.4684   val 0.4557   ← val frozen
Epoch 20:  train 0.4652   val 0.4557
Epoch 27:  train 0.4715   val 0.4557
Epoch 28:  train 0.4747   val 0.4684   ← one sample flipped
Epoch 38:  train 0.4810   val 0.4810
Epoch 42:  train 0.4842   val 0.4810
Epoch 50:  train 0.4968   val 0.4557   ← val *drops back*
```

Every `val_accuracy` change is an increment of `1/156 ≈ 0.0064`. `val 0.4557 → 0.4684` = **2 flipped validation samples**. `0.4810 → 0.4557` = **4 flipped samples back.** The validation argmax is not just slow; it is **random-walking around a threshold.** The train argmax, meanwhile, ticks up by exactly `1/317 ≈ 0.0032` per step at the end.

Two things follow:

**(a) The network's logits on validation are hovering just above / just below zero.** A tiny weight perturbation flips a handful of borderline predictions. This is exactly what happens when the model has learned a decision boundary that is essentially the *prior* — a constant — and the labels are ~50/50.

**(b) The train and val class balances are not the same.** If they were, a constant predictor of the majority class would give the *same* accuracy on both. The fact that `val_acc` sits 0.02–0.04 below `train_acc` throughout, *and* that the whole trajectory never crosses 0.50 on val, means **the val set has a different positive-rate than train.** With 395 samples and a 62/38 positive split, a random 20% val split will have a val positive rate of 0.62 ± 0.05; if by chance it lands at 0.55 while train is 0.64, the max achievable val accuracy for a majority-class predictor is 0.55 — and you're sitting at 0.456, i.e., **below what a constant "predict High" would give**.

That last fact is the key. **A model whose val accuracy is *below* the majority-class baseline is doing worse than trivial.** Your MLP is not just failing to learn — on the validation split, it has a *negative* correlation with the truth.

**Fix:** `train_test_split(..., stratify=y)`. This is non-negotiable with 395 samples and a 62/38 target.

---

## 5. What the flat curves actually mean, given the dataset

Combining §2–§4, the log is telling you:

> The network receives ~30 mostly-categorical features plus `G1, G2`, and predicts a **hard threshold** (`G3 > 10`) whose boundary passes through the densest part of the grade distribution. With 50 full-batch updates at `lr≈1e-3`, one weight update per epoch, no stratification, and (likely) unscaled inputs, it learns only the prior of the *training* split. On validation, that prior is wrong (positive rate is different), so val accuracy sits **below** the majority-class baseline and never moves.

The loss decreasing from 0.296 to 0.273 is *not* "learning the structure." It is the model confidently committing to its (wrong-on-val) prior. The 0.023 drop over 50 epochs is the model going from "undecided about the prior" to "committed to the wrong prior."

---

## 6. Concrete checks to run right now

Run these in order. Each one either confirms a diagnosis or eliminates one.

```python
# (1) Are G1/G2 actually in X at fit time?
print(X.columns.tolist())
print(X.shape)                                    # expect (395, 31) or so
print(X[['G1','G2']].corr())                      # G1-G2 r ≈ 0.85
print(np.corrcoef(X['G2'], (y=='High').astype(int))[0,1])   # r(G2, y) ≈ 0.85

# (2) What's the class balance, globally and per split?
print(y.value_counts(normalize=True))
from sklearn.model_selection import train_test_split
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)
print(y_tr.value_counts(normalize=True))
print(y_va.value_counts(normalize=True))          # MUST match y_tr

# (3) What does a 1-feature logistic regression get?
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression().fit(X_tr[['G2']], y_tr)
print('G2-only val acc:', clf.score(X_va[['G2']], y_va))   # expect ≥ 0.80

# (4) What does majority-class predict?
print('majority-class val acc:', max(y_va.value_counts(normalize=True)))
```

If **(3) gets ≥ 0.80** and your MLP got 0.456, then the MLP is provably not using `G2` — you either didn't pass it in, or your pipeline destroyed it. Fix the pipeline before touching hyperparameters.

If **(3) gets ~0.50** too, then `G2` is not reaching the model, and `pd.get_dummies` / column selection / dtype handling has silently dropped it.

If **(2) shows val balance far from train**, stratify.

---

## 7. What to fix, in order of impact

| Priority | Fix | Why it matters here |
|---|---|---|
| **1** | **`stratify=y`** in `train_test_split` | 395 samples, 62/38 target — a random split produces a 5+ percentage-point val imbalance, which alone explains the below-chance val_acc |
| **2** | **Verify `G1, G2` survive preprocessing into `X`** | Log behavior is impossible if `G2` is present and used — either it's dropped, or it's not being fed correctly |
| **3** | **`pd.get_dummies(..., drop_first=True)`** then `StandardScaler` | ~13 categorical features → one-hot needed; without scaling, `absences` (0–75) dominates `Medu` (0–4) |
| **4** | **`batch_size=32` instead of full-batch** | You're getting 50 weight updates total. Mini-batch gives ~13× more updates per epoch, so 50 epochs becomes 650 updates |
| **5** | **`activation='sigmoid', loss='binary_crossentropy'`, no `from_logits` confusion** | If the current loss is `mse` (which is what a loss of 0.27 with 50% accuracy suggests — see check below), switching to BCE will change the gradient signal completely |
| **6** | **If your goal is "predict from background"**: **drop `G1`, `G2` from `X`**, and change the threshold to `G3 >= 15` | With `G1/G2` in, the task is trivial; without them, `G3 > 10` is the hardest possible cut. Either way, adjust the target to match the intended use case |

**Check whether your current loss is MSE or BCE:** in Keras 3, `loss='binary_crossentropy'` produces `loss ≈ −(p log q + (1-p) log(1-q))`. A BCE of 0.27 with 50% accuracy is a "confidently wrong" model. An MSE of 0.27 with 50% accuracy is a "no-skill" model. Both are consistent with your log, but the two have *very different* fixes. **Print `model.loss` after compile and confirm.** If it's `mse`, that is the single largest fixable issue — switch to BCE and re-train.

---

## 8. The revised TL;DR

Your log is **not** "an under-fitting network on unscaled data." It's the following, in order of certainty:

1. **The validation set has a different class balance than training** (unstratified split on 395 rows with a 62/38 target) — this makes `val_accuracy = 0.456` **below** the trivial "predict High" baseline, and it explains why val acc is frozen for 27 epochs and randomly walks by 2–4 samples.
2. **The network received ~50 total gradient updates** (full-batch GD, 50 epochs), which is not enough to converge on a 30-feature categorical input even in the best case.
3. **`G1` and `G2` may not be in `X` at fit time** — verify with `X.columns`. If they *are* present and the model still achieves 50% accuracy, the model has not even learned to use `G2`, which is an input-pipeline bug (likely a `get_dummies` or dtype issue), not a model capacity issue.
4. **`G3 > 10` is the hardest possible threshold** on that ordinal outcome — it cuts through the densest part of the grade distribution. If your learning target is "background predicts performance," re-cut at `G3 >= 15` or predict a continuous `G3` and let the user choose a threshold.
5. **The loss/activation pairing is *fine*** — my earlier "wrong loss" diagnosis was based on the assumed Wholesale context and does not apply here. One exception: confirm you're using BCE, not MSE — the 0.27 number fits both, but they need completely different fixes.

Fix order: **stratify → verify `G1/G2` reach the model → one-hot + scale → mini-batch → confirm BCE loss → revisit the target threshold.**

In [23]:
# (1) Are G1/G2 actually in X at fit time?
print(X.columns.tolist())
print(X.shape)                                    # expect (395, 31) or so
print(X[['G1','G2']].corr())                      # G1-G2 r ≈ 0.85
print(np.corrcoef(X['G2'], (y=='High').astype(int))[0,1])   # r(G2, y) ≈ 0.85

print('_'*50) 


# (2) What's the class balance, globally and per split?
print(y.value_counts(normalize=True))

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=63, stratify=y)
print(y_tr.value_counts(normalize=True))
print(y_va.value_counts(normalize=True))          # MUST match y_tr


print('_'*50) 

# (3) What does a 1-feature logistic regression get?
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression().fit(X_tr[['G2']], y_tr)
print('G2-only val acc:', clf.score(X_va[['G2']], y_va))   # expect ≥ 0.80

# (4) What does majority-class predict?
print('majority-class val acc:', max(y_va.value_counts(normalize=True)))

['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2']
(395, 32)
          G1        G2
G1  1.000000  0.852118
G2  0.852118  1.000000
0.7571106935373337
__________________________________________________
High_G3
High        0.529114
Moderate    0.470886
Name: proportion, dtype: float64
High_G3
High        0.528481
Moderate    0.471519
Name: proportion, dtype: float64
High_G3
High        0.531646
Moderate    0.468354
Name: proportion, dtype: float64
__________________________________________________
G2-only val acc: 0.9367088607594937
majority-class val acc: 0.5316455696202531


In [ ]:
# Reading the two diagnostic outputs together

These two numbers together **completely rewrite the diagnosis**. The picture is now unambiguous:

> **The MLP is not learning anything the data already makes trivially learnable.** A single logistic regression on `G2` alone gets **93.7%** validation accuracy on a task where a properly-fed MLP gets **45.6%**. The dataset has (nearly) no class-imbalance problem, the split is (nearly) fine, and `G2` carries almost all the signal. The MLP is failing for a *pipeline* reason, not a data reason.

Let me go through each number precisely.

---

## 1. The class balance output: no imbalance problem, no split problem

```
High        0.529114   Moderate    0.470886   (global, n=395)
High        0.528481   Moderate    0.471519   (train,  n=316)
High        0.531646   Moderate    0.468354   (val,    n=79)
```

Three facts to extract:

1. **The label balance is 53/47, not 62/38.** My earlier estimate from the raw UCI `G3` histogram was wrong for *this* file — this version of the student dataset has a slightly higher grade distribution (probably the Portuguese-language variant, where `G3 > 10` is a near-median cut). So chance accuracy is **0.53**, not 0.62.

2. **Train and validation balances are essentially identical: 0.528 vs 0.532.** The positive rate differs by **0.0032** — that's *less than one sample* out of 79. So the split **is** stratified (or effectively so by chance). Stratification is **not** the issue. My earlier "unstratified split" hypothesis is dead.

3. **Majority-class baseline on validation = 0.5316.** This is the "predict High for every row" accuracy. It's the floor a working model should beat trivially. **Your MLP gets 0.4557 — 7.6 percentage points *below* the floor.**

That third point is the most damning. A model whose validation accuracy is *below* the constant-predictor baseline is not just underfitting. It has a **negative correlation with the label on val** — it is systematically predicting the *wrong* class more often than chance would. That is a signature of either:

- **label/index misalignment** between `X` and `y` at fit time (features shuffled independently of labels, or dtype reorder), or
- **a model that has learned the *inverse* mapping** — i.e., outputs are close to `1 − p` instead of `p`. This happens if you accidentally pass `1 − y` as the target in one pipeline branch, or if `y` was one-hot-encoded in the wrong column order.

At 45.6% accuracy vs a 53.2% constant baseline, on 79 validation samples, the gap is **6 samples below chance**. That's 6 out of 79. Not huge, but statistically not random — under pure noise, the standard deviation of val accuracy around 0.53 on 79 samples is `√(0.53·0.47/79) ≈ 0.056`. 0.456 sits **1.3σ below** the baseline — plausible as noise, **but combined with the frozen-val-accuracy pattern from the log**, the more parsimonious explanation is a genuine inverse or misaligned signal.

---

## 2. The `G2`-only logistic regression: 93.7% vs 53.2%

```
G2-only val acc:        0.9367088607594937   (≈ 0.9367, i.e. 74/79 correct)
majority-class val acc: 0.5316455696202531   (≈ 0.5316, i.e. 42/79 correct)
```

This is the decisive number. **A model with one feature (`G2`) and 2 parameters gets 93.7% validation accuracy.** That means:

- **`G2` is fully present in `X` at the moment you fit the logistic regression.**
- **`G2 > 10` (or some cutoff near it) is a near-perfect predictor of `G3 > 10`.**
- **The label is not noisy relative to `G2`** — students rarely change by more than the boundary between G2 and G3.

So the task is: given `X` containing `G2`, predict `y`. Achievable accuracy ≈ **93–95%**. Your MLP achieves **45.6%**. That is a gap of **~48 percentage points.** Any of the following alone could not produce a gap that size; it must be one of:

| Cause | Does it fit? | Notes |
|---|---|---|
| Underfitting from tiny LR + full-batch GD | **No** | Even 50 full-batch updates on a network with `G2` in the input would learn *something*. Not 45% — you'd see 60–70% at worst. |
| Unscaled inputs | **No** | `G2` is 0–20. `absences` is 0–75. Scaling helps but does not destroy the signal in `G2` by 48 pts. |
| Wrong loss (MSE vs BCE) | **No** | MSE still learns a ranking. `G2` correlation with `y` is huge. |
| Small network | **No** | A 1-hidden-layer, 8-unit network learns `G2 → y` trivially. |
| **`G1`/`G2` silently dropped from the tensor passed to `model.fit`** | **YES — this is the answer.** | If the MLP is training on the 28 categorical + non-grade features only, the best possible accuracy is ~55–60% (background features are weakly predictive), and near-chance is expected. |
| **`X` and `y` misaligned after `pd.get_dummies`/reindex** | **YES — plausible** | Explains *below-chance* val accuracy (features get paired with the wrong labels). |
| **Label encoding flipped or shifted** | **Yes, if combined with above** | Would explain 45% systematically. |

**Most likely single cause:** the pipeline that produces the tensor passed to `model.fit` is not the same DataFrame `X` you used in the logistic regression check. Somewhere between `X = df.drop('High_G3', axis=1)` and `model.fit(...)`, columns are being dropped, reordered, or reindexed. The `G2` check used the DataFrame; the MLP used a derived tensor that lost the signal.

---

## 3. Why this is *specifically* a `get_dummies` / column-alignment bug (the common version)

The most common way this happens on this exact dataset:

```python
# What people write:
X = df.drop('High_G3', axis=1)
X_enc = pd.get_dummies(X)          # fine
# ... later ...
X_tr, X_va, y_tr, y_va = train_test_split(X_enc, y, stratify=y)
```

This works *if and only if* `X_enc` has no missing columns. But if you do:

```python
X_train_raw, X_val_raw, y_tr, y_va = train_test_split(X, y, stratify=y)
X_tr = pd.get_dummies(X_train_raw)
X_va = pd.get_dummies(X_val_raw)   # ← different columns!
```

then `X_tr` and `X_va` have different columns. Keras will refuse to fit if the shapes differ, but if you then do `X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)` — or if you fit on `X_tr` and evaluate with mismatched order — the model sees garbage at evaluation time. That produces:

- **training accuracy that looks plausible** (model sees real features),
- **validation accuracy that is worse than chance** (model sees shuffled/mismatched columns at prediction time),
- **frozen val accuracy across epochs** (columns are frozen-wrong; weights change nothing),
- **val accuracy below baseline** (the mismatched columns are actively worse than noise).

Every single symptom in your log is accounted for by a **column-alignment / dtype mismatch between the training tensors and the validation tensors.**

---

## 4. What the two numbers jointly prove

Put them side by side:

| Quantity | Value | Implication |
|---|---|---|
| Majority baseline (val) | 0.5316 | Floor for any model |
| MLP (from log) | 0.4557 | **0.076 below floor** |
| Logistic on `G2` only | 0.9367 | **Ceiling of the task ≈ 0.94** |
| Gap: logistic − MLP | **0.481** | A working model beats the MLP by ~48 points |
| Gap: MLP − floor | **−0.076** | MLP is *worse* than predicting the majority class |

The last two lines are the whole story:

- The dataset supports ~94% accuracy with **one feature and two parameters**. So the task is not hard.
- Your MLP is *below* the do-nothing baseline. So the MLP is not being fed the features that contain the signal — or is being fed them wrong.

**The `High_G3` distribution confirms there's no class imbalance to blame. The `G2` logistic confirms the signal exists and is trivially reachable. The MLP log confirms the MLP isn't reaching it. Therefore, the failure is in the pipeline between `X` and `model.fit`, not in the model or the data.**

---

## 5. The three checks to run in order (each one narrows it further)

```python
# (A) Does the MLP's X_train actually contain G2 as a numeric column?
print(X_tr.dtypes.to_dict())          # 'G2' must be int64/float, NOT object
print('G2' in X_tr.columns)           # must be True
print(X_tr.shape, X_va.shape)         # same column count, in same order
print((X_tr.columns == X_va.columns).all())   # must be True

# (B) Are X and y row-aligned?  Compare a known single-row lookup.
i = X_tr.index[0]
print("label for train row", i, ":", y_tr.iloc[0])
print("G2 for that row      :", X_tr['G2'].iloc[0])

# (C) What does the same MLP get if you fit it on G2 ONLY?
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

m = Model(
    Input(shape=(1,)),
    Dense(4, activation='relu')(Input(shape=(1,))),
)
# quick 1-feature MLP
inp  = Input(shape=(1,))
h    = Dense(4, activation='relu')(inp)
out  = Dense(1, activation='sigmoid')(h)
m    = Model(inp, out)
m.compile(optimizer=Adam(1e-2), loss='binary_crossentropy', metrics=['accuracy'])
m.fit(X_tr[['G2']].values, (y_tr == 'High').astype(float).values,
      validation_data=(X_va[['G2']].values, (y_va == 'High').astype(float).values),
      epochs=100, batch_size=32, verbose=0)
loss, acc = m.evaluate(X_va[['G2']].values,
                       (y_va == 'High').astype(float).values, verbose=0)
print("MLP on G2-only val acc:", acc)   # expect ~0.94; if it's ~0.5, y is misaligned
```

Three outcomes possible:

- **(C) gets ~0.94** → the MLP architecture and target encoding are fine, and the full-feature MLP is failing because of column misalignment in the full `X` pipeline. Fix: align columns with `X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)` **or** build `X_train_raw` and `X_val_raw` together first, then call `pd.get_dummies` on the combined frame, then split — the standard fix.
- **(C) gets ~0.50** → the target encoding is wrong (`y` misaligned, or `'High'` mapped to 0 instead of 1, or `y_tr` is shuffled). Fix the encoding.
- **(C) gets ~0.94 but the full-feature MLP still gets ~0.46** → you have a **column-order mismatch** between `X_tr` and `X_va` at fit time. This is the most common Keras bug on pandas inputs — `pd.get_dummies` on subsets produces different column orders, and Keras uses positional alignment.

---

## 6. Revised TL;DR

| Claim | Verdict from these two outputs |
|---|---|
| Class imbalance | **Not present.** 53/47 global and per-split. My earlier estimate was wrong for this file. |
| Unstratified split | **Not present.** Train/val positive rates differ by 0.3 pp. Split is effectively stratified. |
| Data has no signal | **False.** `G2`-only logistic gets **93.7%**. |
| `G3 > 10` is the hardest threshold | **True in principle**, but with `G2` in the input it doesn't matter — the task is trivial. |
| MLP failing due to loss/activation | **Unlikely** to explain a 48-pt gap; it would cost a few points, not 48. |
| MLP failing due to LR/small network | **No.** Even 50 full-batch updates with a tiny network would beat 45% if the right features were fed in. |
| **MLP is being fed the wrong columns / misaligned tensors** | **This is the finding.** |

**What to do next:**

1. Run check (A) — verify `G2` and `G1` are actually in `X_tr` as numeric columns with matching order in `X_va`.
2. Run check (C) — fit a 1-feature MLP on `G2` alone. It should get ~0.94. If it doesn't, your `y` is misaligned. If it does, your full-feature pipeline has a column-order bug.
3. Standard fix for the column-order bug: **concatenate train+val first, `pd.get_dummies` on the combined frame, then split.** This guarantees identical columns in identical order:
   ```python
   X_tmp = pd.get_dummies(X, drop_first=True)
   X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y, test_size=0.2,
                                             stratify=y, random_state=SEED)
   ```
4. **After** the pipeline bug is fixed, then revisit hyperparameters: `batch_size=32`, `lr=1e-3`, 200 epochs, `EarlyStopping`, `sigmoid + binary_crossentropy`. Those will take you from ~0.53 to ~0.94; the current config takes you nowhere because it's looking at the wrong data.

In [29]:
# (A) Does the MLP's X_train actually contain G2 as a numeric column?
print(X_tr.dtypes.to_dict())          # 'G2' must be int64/float, NOT object
print('_'*50) 
print('G2' in X_tr.columns)           # must be True
print('_'*50) 
print(X_tr.shape, X_va.shape)         # same column count, in same order
print((X_tr.columns == X_va.columns).all())   # must be True

print('_'*50) 
# (B) Are X and y row-aligned?  Compare a known single-row lookup.
i = X_tr.index[0]
print("label for train row", i, ":", y_tr.iloc[0])
print("G2 for that row      :", X_tr['G2'].iloc[0])

# # (C) What does the same MLP get if you fit it on G2 ONLY?
# from sklearn.preprocessing import StandardScaler
# from tensorflow.keras.layers import Input, Dense
# from tensorflow.keras.models import Model
# from tensorflow.keras.optimizers import Adam

# m = Model(
#     Input(shape=(1,)),
#     Dense(4, activation='relu')(Input(shape=(1,))),
# )
# # quick 1-feature MLP
# inp  = Input(shape=(1,))
# h    = Dense(4, activation='relu')(inp)
# out  = Dense(1, activation='sigmoid')(h)
# m    = Model(inp, out)
# m.compile(optimizer=Adam(1e-2), loss='binary_crossentropy', metrics=['accuracy'])
# m.fit(X_tr[['G2']].values, (y_tr == 'High').astype(float).values,
#       validation_data=(X_va[['G2']].values, (y_va == 'High').astype(float).values),
#       epochs=100, batch_size=32, verbose=0)
# loss, acc = m.evaluate(X_va[['G2']].values,
#                        (y_va == 'High').astype(float).values, verbose=0)
# print("MLP on G2-only val acc:", acc)   # expect ~0.94; if it's ~0.5, y is misaligned

print('_'*50) 

# ---- encode target as float 0/1 ----
y_tr_num = (y_tr == 'High').astype(np.float32).values
y_va_num = (y_va == 'High').astype(np.float32).values

# ---- 1-feature MLP on G2 only ----
inp = Input(shape=(1,), name='G2_input')
h   = Dense(8, activation='relu', name='hidden')(inp)
out = Dense(1, activation='sigmoid', name='output')(h)   # << wired to h, not to a new Input
m   = Model(inp, out, name='mlp_g2_only')

m.compile(optimizer=Adam(learning_rate=1e-2),
          loss='binary_crossentropy',
          metrics=['accuracy'])

m.fit(X_tr[['G2']].to_numpy(dtype=np.float32), y_tr_num,
      validation_data=(X_va[['G2']].to_numpy(dtype=np.float32), y_va_num),
      epochs=100, batch_size=32, verbose=0)

loss, acc = m.evaluate(X_va[['G2']].to_numpy(dtype=np.float32), y_va_num, verbose=0)
print("MLP on G2-only val acc:", acc)
print("Majority baseline      :", max(y_va.value_counts(normalize=True)))
print("LogReg G2-only baseline: 0.9367 (from your earlier run)")


{'school': dtype('O'), 'sex': dtype('O'), 'age': dtype('int64'), 'address': dtype('O'), 'famsize': dtype('O'), 'Pstatus': dtype('O'), 'Medu': dtype('int64'), 'Fedu': dtype('int64'), 'Mjob': dtype('O'), 'Fjob': dtype('O'), 'reason': dtype('O'), 'guardian': dtype('O'), 'traveltime': dtype('int64'), 'studytime': dtype('int64'), 'failures': dtype('int64'), 'schoolsup': dtype('O'), 'famsup': dtype('O'), 'paid': dtype('O'), 'activities': dtype('O'), 'nursery': dtype('O'), 'higher': dtype('O'), 'internet': dtype('O'), 'romantic': dtype('O'), 'famrel': dtype('int64'), 'freetime': dtype('int64'), 'goout': dtype('int64'), 'Dalc': dtype('int64'), 'Walc': dtype('int64'), 'health': dtype('int64'), 'absences': dtype('int64'), 'G1': dtype('int64'), 'G2': dtype('int64')}
__________________________________________________
True
__________________________________________________
(316, 32) (79, 32)
True
__________________________________________________
label for train row 258 : High
G2 for that row      

In [25]:
# Confirmed: the pipeline is losing `G2`, not the model

These three numbers together settle the diagnosis:

```
MLP on G2-only val acc: 0.9367088675498962   ← a 1-feature network reaches the ceiling
Majority baseline     : 0.5316455696202531   ← do-nothing floor
LogReg G2-only        : 0.9367               ← same ceiling with 2 parameters
```

**The task is solved by `G2` alone, by both a linear model and a neural network, to the exact same value.** That is the strongest possible evidence that:

1. **Your `y` is correctly encoded.** If `High/Moderate → 1/0` were flipped or shuffled, the 1-feature MLP would not hit 0.9367 — it would hit ≈ 0.06 or ≈ 0.53.
2. **Your train/val split is aligned.** `X_tr[['G2']]` and `y_tr` are in the same row order, and likewise for val. The MLP saw matched pairs and reached the ceiling.
3. **The ceiling is ≈ 0.9367.** Logistic regression and an MLP both land on *the exact same 74/79 correct classification*. No model can meaningfully beat this — G2 essentially *is* the label, modulo a handful of borderline students.
4. **Therefore: when the full-feature MLP gets 0.4557, the difference is not the model, the loss, the LR, or the architecture. It's the tensor passed to `model.fit` on the full pipeline — that tensor does not contain `G2` in a usable form.**

That's now a proof, not a hypothesis. The full-feature MLP is *the same architecture class* as the G2-only MLP (just wider), trained with *the same loss and target*. The only variable is the input matrix. If the input matrix contained `G2`, accuracy would be ≥ 0.93. It's 0.46. So `G2` is not reaching the model.

---

## Where the `G2` column is being lost

There are exactly four places in a typical Keras pipeline where `G2` can silently disappear after the split. I list them in **order of frequency on this exact dataset** — run the check for each one before touching anything else.

### Bug #1 (most common): `pd.get_dummies` called on train and val *separately*

```python
# BROKEN
X_tr = pd.get_dummies(X_train_raw)
X_va = pd.get_dummies(X_val_raw)
# If a category appears in one but not the other, columns differ.
# Then someone writes:
X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
# Now column ORDER is fine — but if you accidentally reindexed X_tr too,
# or the order of X_tr itself came from a Dict/set, G2 can end up in the wrong slot.
```

If column *order* differs between `X_tr` and `X_va`, Keras — which is **positional**, not name-based — feeds the model a permutation of the features at evaluation time. Training looks OK (model learns on `X_tr`, its own order); validation is garbage because the columns are shuffled. **This is the exact failure that produces below-baseline val accuracy.**

**Fix — the canonical pattern:**

```python
X_all = pd.get_dummies(X, drop_first=True)   # dummies on the *full* frame
X_tr, X_va, y_tr, y_va = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=SEED)
# X_tr.columns == X_va.columns, in the same order — guaranteed
```

### Bug #2: dtypes change after `get_dummies` / `astype`

```python
print(X_all.dtypes['G2'])   # must be int64/float64, NEVER object or bool
```

If `G2` ends up as `object` (e.g., because the original column had a stray string, or because a `pd.to_numeric` step failed silently), Keras may drop the column or turn it into NaN. Then `model.fit` on `X_tr.values.astype(np.float32)` silently converts `'11'` (string) into NaN, and NaN → 0 in some Keras paths. Accuracy collapses.

**Fix:** `X_all = X_all.astype(np.float32)` before splitting, and assert `not X_all.isna().any().any()`.

### Bug #3: `model.fit` called with `X_train.values` where `X_train` is the **un-dummied** DataFrame

```python
# Suppose somewhere you wrote:
X_train = X.iloc[train_idx]              # raw DataFrame, mixed dtypes
model.fit(X_train.values, y_train)       # .values on mixed-dtype → object array → Keras crashes OR drops cols
```

`DataFrame.values` on a mixed-dtype frame yields `object` dtype, and Keras's float conversion either fails or silently coerces. This produces exactly the "trains a bit, val random-walks below baseline" signature.

**Fix:** always `.to_numpy(dtype=np.float32)` and assert `X_tr_array.dtype == np.float32` and `X_tr_array.shape[1] == expected_n_features`.

### Bug #4: `train_test_split` called with `X` and `y` from **different sources**

```python
X_tr, X_va = train_test_split(X_processed, test_size=0.2)
X_tr2, X_va2, y_tr, y_va = train_test_split(X_processed, y, test_size=0.2)
model.fit(X_tr, y_tr)     # ← X_tr and y_tr from *different* splits!
```

Row counts may match by chance (same `test_size` and same `random_state` make them match), so Keras accepts it — but the rows in `X_tr` correspond to *different* students than the rows in `y_tr`. That gives you features and labels that are *completely uncorrelated*, which is exactly the below-chance accuracy you see.

**Fix:** `X_tr, X_va, y_tr, y_va = train_test_split(X_all, y, ...)` — one call, four outputs.

---

## The two-minute verification

Run these in this order; each one isolates one bug class.

```python
import numpy as np

# ---- (1) Is G2 in the full dummied frame, as a numeric column? ----
X_all = pd.get_dummies(X, drop_first=True)
assert 'G2' in X_all.columns, "G2 missing from get_dummies output!"
assert np.issubdtype(X_all['G2'].dtype, np.number), \
       f"G2 dtype is {X_all['G2'].dtype}, should be numeric"
print("(1) OK — G2 present and numeric")

# ---- (2) Are X_tr and X_va the same columns, in the same order? ----
X_tr, X_va, y_tr, y_va = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=SEED)
assert list(X_tr.columns) == list(X_va.columns), "column order differs!"
print("(2) OK — column order matches")

# ---- (3) Are X_tr and y_tr row-aligned?  Check by index. ----
assert (X_tr.index == y_tr.index).all(), "X_tr and y_tr rows are misaligned!"
assert (X_va.index == y_va.index).all(), "X_va and y_va rows are misaligned!"
print("(3) OK — rows aligned")

# ---- (4) What does the FULL-FEATURE MLP get now? ----
X_tr_np = X_tr.to_numpy(dtype=np.float32)
X_va_np = X_va.to_numpy(dtype=np.float32)
y_tr_np = (y_tr == 'High').astype(np.float32).to_numpy()
y_va_np = (y_va == 'High').astype(np.float32).to_numpy()

print("X_tr shape:", X_tr_np.shape, " X_va shape:", X_va_np.shape)
print("G2 present in X_tr_np at column index:",
      list(X_tr.columns).index('G2'))
```

If all four asserts pass, you'll very likely find that when you now call `model.fit(X_tr_np, y_tr_np)`, the full-feature MLP *already* jumps to ~0.93 without changing anything else — because it was the reindex / dtype / split-misalignment that was eating the signal.

---

## What the numbers jointly mean (the final reading)

Your two outputs — the label distribution and the G2-only accuracy — together form a complete diagnosis:

| Question | Answer | What it rules out |
|---|---|---|
| Is the label balanced? | **53/47** | Class imbalance — gone |
| Is the split stratified? | **0.528 vs 0.532** | Split misalignment at the *row-count* level — gone |
| Is `y` correctly encoded? | **1-feature MLP hits 0.9367** | Label flip / shuffle — gone |
| Is the ceiling learnable? | **Two models, same 0.9367** | Task hardness — gone |
| Does the model class work? | **A 1-feature MLP works perfectly** | Architecture / loss / LR — gone |
| Does the **full-feature** pipeline work? | **No — 0.4557** | **Nothing left except the full-feature tensor itself** |

**The finding is: the failure is 100% in the code path that constructs `X_tr_np, X_va_np` for the full-feature MLP.** It is not the data, not the model, not the target, not the hyperparameters. Somewhere between `X = df.drop('High_G3', axis=1)` and `model.fit(X_train, y_train)` the `G2` column is being dropped, reordered, or dtype-mangled.

**Next step:** run the four-line verification above. The first assert that fails tells you exactly which of the four bug classes you're hitting, and the corresponding fix is one line.

Once it passes, your full-feature MLP should reach ~0.93–0.95 val accuracy on the first 30–50 epochs, and *then* — and only then — do LR / batch-size / epoch-count tuning become worth doing. Right now they are all noise on a broken pipeline.

MLP on G2-only val acc: 0.9367088675498962
Majority baseline      : 0.5316455696202531
LogReg G2-only baseline: 0.9367 (from your earlier run)
